# Forward Model: Cast Alloys

Handles cast dataset with text mining for composition extraction. Uses hyperparameters from config (run 01_hyperparameter_tuning_forward.ipynb first).

In [1]:
import pandas as pd
import numpy as np
import re
import os
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import warnings

warnings.filterwarnings('ignore')

try:
    from utils import load_hyperparams, load_hyperparams_for_target, get_default_hyperparams
except ImportError:
    import sys
    sys.path.insert(0, os.getcwd())
    from utils import load_hyperparams, load_hyperparams_for_target, get_default_hyperparams

FILE_PATH = 'cleaned_cast_dataset.csv'

In [2]:
# MODULE 1: DATA ENGINEERING & TEXT MINING
class CastAlloyAnalyzer:
    def __init__(self, file_path):
        self.file_path = file_path
        self.raw_data = None
        self.clean_data = None
        self.standard_elements = ['Al', 'Si', 'Fe', 'Cu', 'Mn', 'Mg', 'Cr', 'Ni', 'Zn', 'Ti', 'Ga', 'V']
        self.target_cols = []

    def load_and_repair_data(self):
        print(f"[{'='*25} STEP 1: LOADING & REPAIRING DATA {'='*25}")
        try:
            self.raw_data = pd.read_csv(self.file_path, encoding='utf-8')
        except UnicodeDecodeError:
            self.raw_data = pd.read_csv(self.file_path, encoding='latin-1')
        print(f"Original Data Loaded. Shape: {self.raw_data.shape}")
        extracted_list = self.raw_data['Alloy Name'].apply(self._extract_formula)
        composition_df = pd.DataFrame(list(extracted_list))
        for col in self.standard_elements:
            if col not in composition_df.columns: composition_df[col] = 0.0
        composition_df = composition_df.fillna(0.0)
        df_merged = pd.concat([self.raw_data, composition_df], axis=1)
        self.clean_data = df_merged[df_merged['Al'] > 1.0].reset_index(drop=True)
        print(f"Success: Recovered Composition for {len(self.clean_data)} alloys.")
        exclude = self.standard_elements + ['Series', 'Parent Alloy', 'Alloy Name', 'AlloyNumber', 'Temper', 'Standard']
        self.target_cols = [c for c in self.clean_data.columns if c not in exclude]
        for col in self.target_cols:
            self.clean_data[col] = self.clean_data[col].apply(self._clean_numeric)
        return self.clean_data

    def _extract_formula(self, text):
        if pd.isna(text): return {}
        text = str(text).replace('AI', 'Al')
        m = re.search(r'\((.*?)\)', text)
        if not m: return {}
        formula = m.group(1)
        comp = {}
        for el in self.standard_elements:
            if el == 'Al': continue
            p = re.compile(rf"{el}(\\d*\\.?\\d*)")
            h = p.search(formula)
            if h:
                v = h.group(1)
                comp[el] = float(v) if (v and v != '.') else 0.5
        total = sum(comp.values())
        comp['Al'] = 100 - total if 0 < total < 100 else 0.0
        return comp

    def _clean_numeric(self, val):
        if pd.isna(val): return np.nan
        s = str(val).strip()
        if '-' in s:
            try:
                parts = [re.sub(r'[^0-9.]', '', p) for p in s.split('-')]
                nums = [float(p) for p in parts if p]
                return sum(nums)/len(nums)
            except: pass
        m = re.search(r"[-+]?[0-9]*\.[0-9]+|[0-9]+", s)
        return float(m.group()) if m else np.nan

    def perform_eda(self):
        print(f"\n[{'='*25} STEP 2: EDA {'='*25}")
        top_targets = self.target_cols[:5]
        cols = self.standard_elements[:8] + top_targets
        corr_data = self.clean_data[cols].corr()
        plt.figure(figsize=(12, 8))
        sns.heatmap(corr_data, cmap='coolwarm', linewidths=0.5, annot=False)
        plt.title("Chemical Composition vs. Property Correlation")
        plt.tight_layout()
        plt.show()

In [3]:
# MODULE 2: MODEL TRAINING (with loaded hyperparams)
def build_model(name, params):
    p = params.copy() if params else get_default_hyperparams(name)
    if name == 'XGBoost':
        return xgb.XGBRegressor(objective='reg:squarederror', **{k: v for k, v in p.items()})
    if name == 'RandomForest':
        return RandomForestRegressor(**{k: v for k, v in p.items()})
    if name == 'GradientBoosting':
        return GradientBoostingRegressor(**{k: v for k, v in p.items()})
    return None

class CastModelTrainer:
    def __init__(self, data, input_cols, hyperparams=None):
        self.data = data
        self.inputs = input_cols
        self.best_models = {}
        self.hp = hyperparams or load_hyperparams('cast')

    def train_full_suite(self, target_list):
        print(f"\n[{'='*25} STEP 3: MODEL TRAINING {'='*25}")
        for target in target_list:
            df_t = self.data.dropna(subset=[target])
            if len(df_t) < 10: continue
            X = df_t[self.inputs]
            y = df_t[target]
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            print(f"\n>> Target: {target} (n={len(df_t)})")
            spec = load_hyperparams_for_target('cast', target)
            if spec and spec.get('model') and spec.get('params'):
                name = spec['model']
                params = spec['params']
                model = build_model(name, params)
                model.fit(X_train, y_train)
                pred = model.predict(X_test)
                best_r2 = r2_score(y_test, pred)
                winner = model
                winner_name = name
                mae_c = mean_absolute_error(y_test, pred)
                print(f"   {name} (from config): R²={best_r2:.4f}, MAE={mae_c:.2f}")
            else:
                best_r2 = -float('inf')
                winner = None
                winner_name = None
                print("   (No cast.by_target for this target; comparing XGB/RF/GB)")
                for name in ['XGBoost', 'RandomForest', 'GradientBoosting']:
                    params = self.hp.get(name) if self.hp else get_default_hyperparams(name)
                    model = build_model(name, params)
                    model.fit(X_train, y_train)
                    pred = model.predict(X_test)
                    r2 = r2_score(y_test, pred)
                    mae_m = mean_absolute_error(y_test, pred)
                    print(f"   {name}: R²={r2:.4f}, MAE={mae_m:.2f}")
                    if r2 > best_r2:
                        best_r2 = r2
                        winner = model
                        winner_name = name
            self.best_models[target] = winner
            mae = mean_absolute_error(y_test, winner.predict(X_test))
            print(f"   WINNER: {winner_name} | R²: {best_r2:.4f} | MAE: {mae:.2f}")

    def predict_property(self, composition):
        input_df = pd.DataFrame([composition])
        for col in self.inputs:
            if col not in input_df.columns: input_df[col] = 0.0
        input_df = input_df[self.inputs]
        print(f"\n[{'='*25} PREDICTION {'='*25}")
        print(f"Alloy Recipe: {composition}")
        print("-" * 60)
        for target, model in self.best_models.items():
            val = model.predict(input_df)[0]
            print(f"{target:<40} : {val:.2f}")

In [4]:
# MAIN EXECUTION
analyzer = CastAlloyAnalyzer(FILE_PATH)
df_clean = analyzer.load_and_repair_data()

if len(df_clean) > 0:
    analyzer.perform_eda()
    trainer = CastModelTrainer(df_clean, analyzer.standard_elements)
    targets_to_train = [t for t in analyzer.target_cols if 'Strength' in t or 'Conductivity' in t or 'Modulus' in t]
    if not targets_to_train: targets_to_train = analyzer.target_cols[:10]
    trainer.train_full_suite(targets_to_train)
    sample_alloy = {'Al': 89.5, 'Si': 10.0, 'Mg': 0.5, 'Fe': 0.0, 'Cu': 0.0}
    trainer.predict_property(sample_alloy)
else:
    print("Could not extract enough data to proceed.")

[========================= STEP 1: LOADING & REPAIRING DATA =========================
Original Data Loaded. Shape: (110, 32)
Success: Recovered Composition for 0 alloys.
Could not extract enough data to proceed.
